# Satellite-to-Satellite Validation: COSMIC-2

This notebook validates TAT-C's satellite-to-satellite analysis (`collect_space_observations`, see the [Space-Based Coverage](../examples/CollectSpace.ipynb) example) against the radio occultations recorded by the [COSMIC-2](https://www.cosmic.ucar.edu/what-we-do/cosmic-2) constellation on 14 May 2026. `collect_space_observations` finds the periods when one satellite can observe another, as by a sensor or a radio link, subject to the slant range and to the Earth's occlusion of the straight line of sight between them: its minimum WGS 84 geodetic altitude must be at least `min_grazing_altitude`.

A radio occultation (RO) is such an occlusion event. As a GNSS satellite sets behind the Earth as seen from a receiver in low Earth orbit (or rises from behind it), the line of sight between them sweeps down (or up) through the atmosphere, which the receiver profiles from the bending of the signal. TAT-C's RO analysis (`collect_ro_observations`, validated in [Radio Occultation Validation: COSMIC-2](ValidateROCOSMIC2.ipynb)) reports each occultation at the time the straight line reaches its `sample_elevation`, by default -80 km (refraction is not modeled). With `min_grazing_altitude` set to the same -80 km, each window of `collect_space_observations` from a receiver to a transmitter should therefore start with a rising occultation and end with a setting one. The satellites' recorded positions provide an independent truth for the occlusion constraint, the search for window edges, and the analysis of many pairs of satellites at once:

1. **Observed occultations.** Is every occultation COSMIC-2 recorded at the edge of a window: a start if rising, an end if setting?
2. **Timing.** How closely do the edges match the true times, from the recorded positions?
3. **Consistency with the RO analysis.** Do the edges match the occultations of `collect_ro_observations` one for one?
4. **Symmetry, resolution, and cost.** Are the windows the same in both directions and for any `min_duration`, and how long does the analysis take?

## Reference Data

This notebook reuses the data downloaded (and cached in `data/ro`) by the [COSMIC-2 RO validation](ValidateROCOSMIC2.ipynb), which must be run first: the [AWS Registry of Open Data](https://registry.opendata.aws/gnss-ro-opendata/) metadata database, which lists the 5,710 occultations COSMIC-2 recorded on 14 May 2026 with their receiver, transmitter, and rising or setting flag, and the **calibratedPhase** (Level 1A) files of the same seeded random sample of 200 occultations from each constellation (GPS, GLONASS, and Galileo), which record the receiver and transmitter positions at 100 Hz. The orbits are the two-line element sets (TLEs) included with that notebook (`gps_2026-05-14.3le`, `glonass_2026-05-14.3le`, `galileo_2026-05-14.3le`, and `cosmic2_2026-05-14.3le`, from [Space-Track](https://www.space-track.org)).

In [1]:
import os

import numpy as np
from awsgnssroutils.database import RODatabaseClient

CONSTELLATIONS = {"G": "GPS", "R": "GLONASS", "E": "Galileo"}

DATA_DIR = os.path.join("data", "ro")
if not os.path.isdir(os.path.join(DATA_DIR, "metadata")):
    raise FileNotFoundError("run ValidateROCOSMIC2 first, to download the COSMIC-2 occultation database")
rodb = RODatabaseClient(metadata_root=os.path.join(DATA_DIR, "metadata"), version="v1.1")
occultations = rodb.query(
    missions="cosmic2", datetimerange=("2026-05-14", "2026-05-14T23:59:59"), silent=True
)
# the RO validation's seeded random sample of 200 occultations with a retrieval from each constellation
sample = occultations[0:0]
sample._data = []
for k, code in enumerate(CONSTELLATIONS):
    pool = occultations.filter(
        GNSSconstellations=code,
        availablefiletypes=["ucar_calibratedPhase", "ucar_refractivityRetrieval"],
    )
    rng = np.random.default_rng(20260514 + k)
    sample._data += [pool._data[i] for i in sorted(rng.choice(pool.size, 200, replace=False))]
sample.size = len(sample._data)
missing = [
    r["occid"]
    for r in sample._data
    if not os.path.exists(os.path.join(DATA_DIR, os.path.basename(r["ucar_calibratedPhase"])))
]
if missing:
    raise FileNotFoundError(f"run ValidateROCOSMIC2 first, to download {len(missing)} calibratedPhase files")
print(f"{occultations.size} COSMIC-2 occultations on 14 May 2026, {sample.size} sampled")

5710 COSMIC-2 occultations on 14 May 2026, 600 sampled


## Orbits

The receivers and transmitters are identified, and poor element sets excluded, exactly as in the COSMIC-2 RO validation (see its Test 1): each is matched to the satellite whose element sets best reproduce the positions recorded in the sampled Level 1A files, and element sets more than 20 km from those positions are excluded. This takes about a minute and a half.

In [2]:
import collections
from datetime import datetime, timedelta, timezone

import netCDF4 as nc
import pandas as pd
from joblib import Parallel, delayed

from tatc.constants import timescale
from tatc.schemas import GeneralPerturbationsOrbit, Satellite

DAY0 = datetime(2026, 5, 14, tzinfo=timezone.utc)
GPS_EPOCH = datetime(1980, 1, 6, tzinfo=timezone.utc)
GPS_UTC_OFFSET = 18  # s, GPS time minus UTC in 2026
MAX_ERROR = 20  # km, largest acceptable position error of an element set
N_JOBS = min(16, os.cpu_count())  # parallel worker processes

# element sets per NORAD catalog number, keyed by epoch (later duplicates win)
tle_lines = [
    line.rstrip()
    for path in ["gps_2026-05-14.3le", "glonass_2026-05-14.3le", "galileo_2026-05-14.3le", "cosmic2_2026-05-14.3le"]
    for line in open(path)
    if line.strip()
]
element_sets, names = collections.defaultdict(dict), {}
for i in range(0, len(tle_lines), 3):
    line1, line2 = tle_lines[i + 1], tle_lines[i + 2]
    names[line1[2:7]] = tle_lines[i][2:].strip()
    element_sets[line1[2:7]][float(line1[18:32])] = [line1, line2]


def to_orbit(sets):
    return GeneralPerturbationsOrbit.from_tle(sum((sets[e] for e in sorted(sets)), []))


def identify(positions, element_sets, max_error):
    """Matches recorded Earth-fixed positions [(datetime, m)] to the satellite whose element sets
    reproduce them best, keeping that satellite's element sets within max_error (km) of all of them.
    Returns the NORAD catalog number, kept element sets, their position errors (km), the next-best
    satellite's median error (km), and the number of excluded element sets."""
    import numpy as np
    from skyfield.framelib import itrs

    from tatc.constants import timescale
    from tatc.schemas import GeneralPerturbationsOrbit

    ts = timescale.from_datetimes([t for t, _ in positions])
    rotation = itrs.rotation_at(ts)
    recorded_m = np.array([p for _, p in positions]).T

    def errors(sets):
        orbit = GeneralPerturbationsOrbit.from_tle(sum((sets[e] for e in sorted(sets)), []))
        p = np.einsum("ij...,j...->i...", rotation, np.array(orbit.get_orbit_track_at_time(ts).position.m))
        return np.linalg.norm(p - recorded_m, axis=0) / 1e3

    best = {norad: min(np.median(errors({e: s})) for e, s in sets.items()) for norad, sets in element_sets.items()}
    norad = min(best, key=best.get)
    kept = {e: s for e, s in element_sets[norad].items() if errors({e: s}).max() <= max_error}
    return norad, kept, errors(kept) if kept else None, sorted(best.values())[1], len(element_sets[norad]) - len(kept)


# recorded positions of each receiver and transmitter
recorded = collections.defaultdict(list)
for record in sample._data:
    l1 = nc.Dataset(os.path.join(DATA_DIR, os.path.basename(record["ucar_calibratedPhase"])))
    l1.set_auto_mask(False)
    for k in [0, l1["time"].size // 2, l1["time"].size - 1]:
        t = GPS_EPOCH + timedelta(seconds=float(l1["startTime"][...]) - GPS_UTC_OFFSET + float(l1["time"][k]))
        recorded[record["transmitter"]].append((t, l1["positionGNSS"][k].astype(float)))
        recorded[record["receiver"]].append((t, l1["positionLEO"][k].astype(float)))


keys = sorted(recorded)
results = Parallel(n_jobs=N_JOBS)(delayed(identify)(recorded[key], element_sets, MAX_ERROR) for key in keys)
rows, satellites = [], {}
for key, (norad, kept, errors, next_best, excluded) in zip(keys, results):
    row = {"key": key, "NORAD": norad, "name": names[norad], "element sets": len(kept), "excluded sets": excluded}
    if kept:  # identified: at least one element set reproduces the recorded positions
        satellites[key] = Satellite(name=key, orbit=to_orbit(kept))
        row.update({"median error (km)": np.median(errors), "max error (km)": errors.max()})
    row["next-best satellite (km)"] = next_best
    rows.append(row)
orbit_check = pd.DataFrame(rows).set_index("key")
orbit_check["group"] = [CONSTELLATIONS.get(k[0], "receivers") for k in keys]
receivers = [satellites[k] for k in sorted(satellites) if k.startswith("cosmic2")]
transmitters = [satellites[k] for k in sorted(satellites) if k[0] in CONSTELLATIONS]
unidentified = sorted(k for k in recorded if k not in satellites)
print(f"{len(receivers)} receivers and {len(transmitters)} transmitters identified; unidentified: {unidentified or 'none'}")
orbit_check.groupby("group", sort=False).agg(
    **{
        "satellites": ("NORAD", "size"),
        "median error (km)": ("median error (km)", "median"),
        "max error (km)": ("max error (km)", "max"),
        "excluded sets": ("excluded sets", "sum"),
        "closest next-best satellite (km)": ("next-best satellite (km)", "min"),
    }
).round(2)

5 receivers and 77 transmitters identified; unidentified: none


,satellites,median error (km),max error (km),excluded sets,closest next-best satellite (km)
group,,,,,
Galileo,26,1.70,15.67,0,7121.83
GPS,32,1.28,4.66,1,2151.02
GLONASS,19,1.23,6.88,0,3901.41
receivers,5,0.53,1.19,0,3338.26


As in the RO validation, all 5 receivers and 77 transmitters (32 GPS, 19 GLONASS, and 26 Galileo satellites) are identified, and one poor GPS element set is excluded.

## Windows

`collect_space_observations` is run from the 5 receivers to the 77 transmitters over the day, with a `min_grazing_altitude` of -80 km, no range limits, and the default `min_duration` (60 s). Within the day, each window's start is a rising occultation, when the line of sight rises above -80 km, and each window's end a setting one; windows that span the start or end of the day are cut off there.

In [3]:
import time

from tatc.analysis import collect_space_observations

SAMPLE_ELEVATION = -80e3  # m, collect_ro_observations' default sample_elevation
DAY1 = DAY0 + timedelta(days=1)

tic = time.perf_counter()
links = collect_space_observations(
    receivers, DAY0, DAY1, to_satellites=transmitters, min_grazing_altitude=SAMPLE_ELEVATION
)
elapsed = time.perf_counter() - tic
# window edges within the day: a start is a rising occultation, an end a setting one
edges = pd.concat(
    [
        pd.DataFrame(
            {
                "receiver": links.from_satellite,
                "transmitter": links.to_satellite,
                "time": links[key],
                "rising": key == "start",
            }
        )
        for key in ["start", "end"]
    ],
    ignore_index=True,
)
edges = edges[(edges.time > DAY0) & (edges.time < DAY1)].sort_values("time", ignore_index=True)


def nearest_edges(events, tolerance):
    """The nearest window edge (index in edges, or -1 if none within tolerance, in s) of the same receiver
    and transmitter to each event (by its time), and the time from the event to the edge (s)."""
    index, offset = np.full(len(events), -1), np.full(len(events), np.nan)
    groups = edges.groupby(["receiver", "transmitter"]).indices
    for key, rows in events.groupby(["receiver", "transmitter"]).indices.items():
        if key not in groups:
            continue
        candidates = groups[key]
        dt = (edges.time.values[candidates][None, :] - events.time.values[rows][:, None]) / np.timedelta64(1, "s")
        j = np.argmin(np.abs(dt), axis=1)
        d = dt[np.arange(len(rows)), j]
        ok = np.abs(d) <= tolerance
        index[rows[ok]] = candidates[j[ok]]
        offset[rows[ok]] = d[ok]
    return index, offset


print(
    f"{len(links)} windows of {len(receivers) * len(transmitters)} receiver-transmitter pairs found in {elapsed:.1f} s, "
    f"with {len(edges)} edges within the day ({edges.rising.sum()} rising, {(~edges.rising).sum()} setting)"
)

5265 windows of 385 receiver-transmitter pairs found in 1.4 s, with 10073 edges within the day (5034 rising, 5039 setting)


## Test 1: Observed Occultations

Each occultation in the database is paired with the nearest window edge of the same receiver and transmitter within 10 minutes of its time stamp. The time stamp (the minute in the `occid`) marks the start of the record for a setting occultation and its end for a rising one, when the straight line is near 130 km for GPS (see the RO validation). A setting occultation's edge (a window's end) should therefore follow its time stamp, and a rising occultation's edge (a window's start) precede it, by one to a few minutes.

In [4]:
observed = pd.DataFrame(
    [
        {
            "occid": r["occid"],
            "receiver": r["receiver"],
            "transmitter": r["transmitter"],
            "constellation": CONSTELLATIONS[r["transmitter"][0]],
            "time": datetime.fromisoformat(r["time"]).replace(tzinfo=timezone.utc),
            "setting": bool(r["setting"]),
        }
        for r in occultations._data
        if r["transmitter"][0] in CONSTELLATIONS
    ]
)
observed["edge"], observed["offset"] = nearest_edges(observed, 600)
at_edge = observed.edge >= 0
observed["right kind"] = at_edge & (edges.rising.values[observed.edge.clip(lower=0)] != observed.setting)
observed["direction"] = observed.setting.map({True: "setting", False: "rising"})
print(f"{len(observed)} observed occultations; without a window edge within 10 minutes:")
display(observed.loc[~at_edge, ["occid", "direction", "time"]])
observed[at_edge].groupby(["constellation", "direction"], sort=False).agg(
    **{
        "at an edge": ("occid", "size"),
        "of the right kind": ("right kind", "sum"),
        "edge minus time stamp, 1% (s)": ("offset", lambda x: x.quantile(0.01)),
        "50% (s)": ("offset", "median"),
        "99% (s)": ("offset", lambda x: x.quantile(0.99)),
    }
).round(1)

5710 observed occultations; without a window edge within 10 minutes:


,occid,direction,time
630,cosmic2e4-R07-202605140000,rising,2026-05-14 00:00:00+00:00
666,cosmic2e5-G22-202605140002,rising,2026-05-14 00:02:00+00:00
1085,cosmic2e6-R16-202605140001,rising,2026-05-14 00:01:00+00:00
1263,cosmic2e6-G12-202605140001,rising,2026-05-14 00:01:00+00:00
2014,cosmic2e1-R18-202605142359,setting,2026-05-14 23:59:00+00:00
4275,cosmic2e4-G21-202605142359,setting,2026-05-14 23:59:00+00:00
4458,cosmic2e2-G01-202605140000,rising,2026-05-14 00:00:00+00:00
4532,cosmic2e1-R20-202605140002,rising,2026-05-14 00:02:00+00:00
5133,cosmic2e2-G24-202605142358,setting,2026-05-14 23:58:00+00:00
5209,cosmic2e6-G13-202605140000,rising,2026-05-14 00:00:00+00:00


at an edge  of the right kind  \
constellation direction                                  
GLONASS       setting           955                955   
              rising            866                866   
GPS           setting          1485               1485   
              rising           1199               1199   
Galileo       setting           980                980   
              rising            214                214   

                         edge minus time stamp, 1% (s)  50% (s)  99% (s)  
constellation direction                                                   
GLONASS       setting                             68.0    117.7    213.7  
              rising                            -227.0   -113.8    -62.7  
GPS           setting                             68.8    116.9    209.7  
              rising                            -207.6   -111.6    -63.8  
Galileo       setting                             68.2    115.7    220.6  
              rising                            -211.0   -111.8    -63.4

All 5,699 occultations whose straight line reaches -80 km within the day are at a window edge of the right kind: setting occultations at a window's end, 68 to 221 s after their time stamp, and rising ones at a window's start, 63 to 227 s before it (1st to 99th percentiles, for every constellation). The other 11 were recorded within 2 minutes of midnight, so their straight lines reach -80 km on the previous or next day, outside the analysis; the RO validation excludes them too.

## Test 2: Timing

For each sampled occultation, the true time at which the straight line reaches -80 km is computed from the recorded receiver and transmitter positions (subsampled to 10 Hz) with TAT-C's tangent point routine, as in the RO validation's Test 4, and paired with the nearest window edge within 60 s. To explain the differences, the along-track errors of the receiver's and transmitter's orbits are also computed at the middle of each occultation, from the recorded positions.

In [5]:
from pyproj import Transformer
from skyfield.framelib import itrs

from tatc.utils import compute_tangent_point

TO_LLA = Transformer.from_crs("EPSG:4978", "EPSG:4979", always_xy=True)


def crossing(t, h, value):
    """Time at which h(t) first crosses value (linear interpolation), or NaN."""
    c = np.nonzero(np.diff(np.sign(h - value)))[0]
    if len(c) == 0:
        return np.nan
    j = c[0]
    return t[j] + (value - h[j]) / (h[j + 1] - h[j]) * (t[j + 1] - t[j])


def along_track_error(satellite, t, recorded):
    """Along-track error (km) of a satellite's orbit at datetime t, from its recorded Earth-fixed position (m)."""
    ts = timescale.from_datetimes([t, t + timedelta(seconds=1)])
    p = np.einsum(
        "ij...,j...->i...", itrs.rotation_at(ts), satellite.orbit.to_gp_orbit().get_orbit_track_at_time(ts).position.m
    )
    direction = (p[:, 1] - p[:, 0]) / np.linalg.norm(p[:, 1] - p[:, 0])
    return (p[:, 0] - recorded) @ direction / 1e3


# true time at which each sampled occultation's straight line reaches -80 km, from the recorded receiver
# and transmitter positions (subsampled to 10 Hz), and the orbits' along-track errors at mid-occultation
rows = []
for record in sample._data:
    l1 = nc.Dataset(os.path.join(DATA_DIR, os.path.basename(record["ucar_calibratedPhase"])))
    l1.set_auto_mask(False)
    t, rx, tx = l1["time"][:], l1["positionLEO"][:].T, l1["positionGNSS"][:].T
    ok = (np.abs(t) < 1e10) & np.all(np.abs(rx) < 1e10, axis=0) & np.all(np.abs(tx) < 1e10, axis=0)
    i = np.nonzero(ok)[0][::10]
    start = float(l1["startTime"][...])
    # GPS seconds since 1980-01-06 are TAI - 19 s
    heights = TO_LLA.transform(*compute_tangent_point(rx[:, i], tx[:, i] - rx[:, i]))[2]
    t_80 = crossing(t[i], np.asarray(heights), SAMPLE_ELEVATION)
    k = i[len(i) // 2]
    t_mid = GPS_EPOCH + timedelta(seconds=start - GPS_UTC_OFFSET + float(t[k]))
    rows.append(
        {
            "occid": record["occid"],
            "receiver": record["receiver"],
            "transmitter": record["transmitter"],
            "constellation": CONSTELLATIONS[record["transmitter"][0]],
            "setting": bool(record["setting"]),
            "time": GPS_EPOCH + timedelta(seconds=start - GPS_UTC_OFFSET + t_80) if np.isfinite(t_80) else pd.NaT,
            "receiver along-track error (km)": along_track_error(satellites[record["receiver"]], t_mid, rx[:, k]),
            "transmitter along-track error (km)": along_track_error(satellites[record["transmitter"]], t_mid, tx[:, k]),
        }
    )
truth = pd.DataFrame(rows)
truth["time"] = pd.to_datetime(truth.time, utc=True)
in_day = (truth.time >= DAY0) & (truth.time < DAY1)
print(f"{len(truth)} sampled occultations, {int(in_day.sum())} reaching -80 km within the day")
truth = truth[in_day].reset_index(drop=True)
truth["edge"], truth["offset"] = nearest_edges(truth, 60)
truth["right kind"] = (truth.edge >= 0) & (edges.rising.values[truth.edge.clip(lower=0)] != truth.setting)
truth["abs offset"] = truth.offset.abs()
truth.groupby("constellation", sort=False).agg(
    **{
        "sampled": ("occid", "size"),
        "at an edge of the right kind": ("right kind", "sum"),
        "mean edge minus truth (s)": ("offset", "mean"),
        "median |edge minus truth| (s)": ("abs offset", "median"),
        "95% (s)": ("abs offset", lambda x: x.quantile(0.95)),
        "max (s)": ("abs offset", "max"),
        "transmitter along-track error, median (km)": ("transmitter along-track error (km)", "median"),
        "receiver along-track error, median (km)": ("receiver along-track error (km)", "median"),
    }
).round(3)

600 sampled occultations, 594 reaching -80 km within the day


,sampled,at an edge of the right kind,mean edge minus truth (s),median |edge minus truth| (s),95% (s),max (s),"transmitter along-track error, median (km)","receiver along-track error, median (km)"
constellation,,,,,,,,
GPS,199,199,-0.053,0.085,0.249,0.499,-0.326,0.061
GLONASS,198,198,-0.019,0.074,0.252,0.409,-0.103,0.011
Galileo,197,197,-0.062,0.093,0.295,0.554,-1.037,0.051


Every sampled occultation is at a window edge of the right kind, within 0.25 to 0.30 s of the true time (95th percentile of the absolute difference, by constellation). On average the edges come 0.02 s (GLONASS) to 0.06 s (Galileo) early, for rising and setting occultations alike:

In [6]:
truth.assign(direction=truth.setting.map({True: "setting", False: "rising"})).pivot_table(
    index="constellation", columns="direction", values="offset", aggfunc="mean", sort=False
).rename_axis(columns="mean edge minus truth (s)").round(3)

mean edge minus truth (s),setting,rising
constellation,,
GPS,-0.045,-0.062
GLONASS,-0.001,-0.036
Galileo,-0.059,-0.080


The bias has the same sign for rising and setting occultations, so it is not an offset in the altitude of the line of sight, which would shift rising and setting edges in opposite directions. Nor does it come from the orbits: the receivers' orbits are accurate along track (median errors of 0.01 to 0.06 km), and the [PlanetiQ RO validation](ValidateROPlanetiQ.ipynb), with the same transmitter orbits, finds no bias (0.00 to 0.02 s). It appears specific to COSMIC-2's recorded times or positions; its cause is not identified here.

`collect_ro_observations` reports the same times (Test 3), and the RO validation's Test 4 finds the same differences.

## Test 3: Consistency with the RO Analysis

`collect_ro_observations` is run for the same receivers and transmitters with a 1 s `time_step` and a `max_yaw` of 90 deg, which removes the receiver antenna's yaw limit, so that it reports every occultation the geometry allows. It finds each RO observation's arc from its tangent point track, samples the arc every second within an elevation range (by default -200 to 60 km), and interpolates the time at -80 km (`time`); `collect_space_observations` instead finds the window edges directly, from the minimum altitude of the line of sight. Each RO `time` is paired with the nearest window edge of the same receiver and transmitter within 1 s. The run takes about a minute in parallel worker processes, and its results are cached in `data/ro`.

In [7]:
import shapely

from tatc.analysis import collect_ro_observations


def folded(yaw):
    """Yaw angle from the nearer of the forward and rearward directions (deg)."""
    yaw = np.abs(np.asarray(yaw, dtype=float))
    return np.minimum(yaw, 180 - yaw)


cache = os.path.join(DATA_DIR, "predicted_cosmic2_yaw90_gnss.pkl")
if not os.path.exists(cache):
    # one parallel task per receiver, all transmitters together
    results = Parallel(n_jobs=N_JOBS)(
        delayed(collect_ro_observations)(
            receiver, transmitters, DAY0, DAY1, time_step=timedelta(seconds=1), max_yaw=90
        )
        for receiver in receivers
    )
    pd.concat(results, ignore_index=True).to_pickle(cache)
ro = pd.read_pickle(cache)
ro["edge"], ro["offset"] = nearest_edges(ro, 1)
paired = ro[ro.edge >= 0]
if paired.edge.duplicated().any():
    raise ValueError("an edge is paired with more than one RO occultation")
# tangent point elevations of each RO observation's samples
elevations = ro.geometry.apply(lambda g: shapely.get_coordinates(g, include_z=True)[:, 2])
crosses = elevations.apply(lambda z: z.min() <= SAMPLE_ELEVATION <= z.max())
truncated = (ro.start == DAY0) | (ro.end == DAY1)
unpaired = ro.edge < 0
# edges without an RO occultation, and whether each falls within an RO observation paired with another edge
unpaired_edges = edges.drop(paired.edge)
within = [
    ((paired.receiver == e.receiver) & (paired.transmitter == e.transmitter) & (paired.start <= e.time) & (paired.end >= e.time)).any()
    for e in unpaired_edges.itertuples()
]
disagree = paired[paired.is_rising.astype(bool).values != edges.rising.values[paired.edge]]
print(f"{len(ro)} RO occultations (max_yaw = 90 deg), {len(edges)} window edges")
print(f"paired one to one within 1 s: {len(paired)}; |RO time minus edge| median {1e3 * paired.offset.abs().median():.2f} ms, max {1e3 * paired.offset.abs().max():.2f} ms")
print(f"RO occultations without an edge: {int(unpaired.sum())}")
print(f"  cut off at the start or end of the day: {int((unpaired & truncated).sum())}")
print(f"  not cut off, straight line never crossing -80 km: {int((unpaired & ~truncated & ~crosses).sum())} (lowest elevations, km: {', '.join(f'{z.min() / 1e3:.0f}' for z in elevations[unpaired & ~truncated].sort_values(key=lambda s: s.apply(min)))})")
print(f"  other: {int((unpaired & ~truncated & crosses).sum())}")
print(f"edges without an RO occultation: {len(unpaired_edges)}, of which within an RO observation paired with another edge: {sum(within)}")
abeam = folded(paired.rx_tx_yaw) > 75
print(
    f"paired occultations disagreeing on direction: {len(disagree)} of {len(paired)} "
    f"({int(disagree.index.isin(paired.index[abeam]).sum())} of the {int(abeam.sum())} more than 75 deg from fore/aft)"
)

10058 RO occultations (max_yaw = 90 deg), 10073 window edges
paired one to one within 1 s: 10019; |RO time minus edge| median 0.14 ms, max 0.93 ms
RO occultations without an edge: 39
  cut off at the start or end of the day: 23
  not cut off, straight line never crossing -80 km: 16 (lowest elevations, km: -200, -79, -79, -76, -72, -62, -37, -21, 20, 25, 33, 37, 42, 48, 57, 60)
  other: 0
edges without an RO occultation: 54, of which within an RO observation paired with another edge: 54
paired occultations disagreeing on direction: 0 of 10019 (0 of the 311 more than 75 deg from fore/aft)


The two analyses agree: 10,019 RO occultations and window edges pair one to one, with times within 1 ms (median 0.14 ms), and agree on the direction (rising or setting) of every one, including those nearly abeam of the receiver. The others reflect differences in what each analysis reports, not in the geometry:

* **Edges without an RO occultation (54).** Each is a second crossing of -80 km within an RO observation already paired with another edge. In grazing geometries, nearly side-on to the receiver, the straight line dips below -80 km and rises again (or the reverse) within the RO elevation range. `collect_ro_observations` reports one `time` for the observation, at the first crossing, where `collect_space_observations` reports both edges.
* **RO occultations without an edge (39).** 23 are cut off at the start or end of the day before reaching -80 km. The other 16 are grazing occultations whose straight line never crosses -80 km: 15 turn back above it (at lowest, between -79 and 60 km) and one stays below it. As documented, `collect_ro_observations` reports each with a `time` and `position` at the endpoint of its profile nearest -80 km, rather than at -80 km.

`collect_ro_observations` determines the direction from the tangent point's profile through -80 km. An earlier approximation, from the transmitter's pitch at the observation's first sample, mislabeled 20 of these occultations (0.2%), all 75 to 90 deg from the receiver's forward or rearward direction: outside COSMIC-2's yaw limit of 65 deg, but within PlanetiQ's (85 deg). This comparison exposed it.

## Test 4: Symmetry, Resolution, and Cost

The line-of-sight constraints are symmetric, so the windows from the transmitters to the receivers should be the same as those from the receivers to the transmitters. All 82 satellites are also analyzed together (with `to_satellites` omitted), as 6,642 ordered pairs, including those between receivers and between transmitters, of which the receiver-transmitter windows should again be the same. Finally, `min_duration` sets the sampling of the search for window edges (at half its value): windows shorter than `min_duration` may be missed, but longer ones should be found at any value.

In [8]:
def windows(frame, reverse=False):
    """Windows by receiver and transmitter (observed and observing satellites, if reverse) and start."""
    columns = ["to_satellite", "from_satellite"] if reverse else ["from_satellite", "to_satellite"]
    return (
        frame[columns + ["start", "end"]]
        .set_axis(["receiver", "transmitter", "start", "end"], axis=1)
        .sort_values(["receiver", "transmitter", "start"], ignore_index=True)
    )


def difference(a, b):
    """Largest difference of window start or end (s) between two sets of windows, or None if they differ in number."""
    if len(a) != len(b) or not (a[["receiver", "transmitter"]] == b[["receiver", "transmitter"]]).all().all():
        return None
    return max((a.start - b.start).abs().max(), (a.end - b.end).abs().max()).total_seconds()


rows = []
tic = time.perf_counter()
reverse = collect_space_observations(
    transmitters, DAY0, DAY1, to_satellites=receivers, min_grazing_altitude=SAMPLE_ELEVATION
)
rows.append(
    {
        "analysis": "transmitters to receivers",
        "pairs": len(transmitters) * len(receivers),
        "windows": len(reverse),
        "receiver-transmitter windows": len(reverse),
        "largest difference (s)": difference(windows(links), windows(reverse, reverse=True)),
        "time (s)": time.perf_counter() - tic,
    }
)
tic = time.perf_counter()
everything = collect_space_observations(receivers + transmitters, DAY0, DAY1, min_grazing_altitude=SAMPLE_ELEVATION)
elapsed = time.perf_counter() - tic
subset = everything[everything.from_satellite.str.startswith("cosmic2") & everything.to_satellite.str[0].isin(list(CONSTELLATIONS))]
n = len(receivers) + len(transmitters)
rows.append(
    {
        "analysis": "all satellites, both directions",
        "pairs": n * (n - 1),
        "windows": len(everything),
        "receiver-transmitter windows": len(subset),
        "largest difference (s)": difference(windows(links), windows(subset)),
        "time (s)": elapsed,
    }
)
for seconds in [5, 20, 60, 300, 1200]:
    tic = time.perf_counter()
    result = collect_space_observations(
        receivers,
        DAY0,
        DAY1,
        to_satellites=transmitters,
        min_grazing_altitude=SAMPLE_ELEVATION,
        min_duration=timedelta(seconds=seconds),
    )
    rows.append(
        {
            "analysis": f"min_duration = {seconds} s",
            "pairs": len(receivers) * len(transmitters),
            "windows": len(result),
            "receiver-transmitter windows": len(result),
            "largest difference (s)": difference(windows(links), windows(result)),
            "time (s)": time.perf_counter() - tic,
        }
    )
inner = links[(links.start > DAY0) & (links.end < DAY1)]
print(f"shortest window within the day: {(inner.end - inner.start).min().total_seconds():.0f} s")
pd.DataFrame(rows).set_index("analysis").round(3)

shortest window within the day: 462 s


,pairs,windows,receiver-transmitter windows,largest difference (s),time (s)
analysis,,,,,
transmitters to receivers,385,5265,5265,0.000,1.330
"all satellites, both directions",6642,20602,5265,0.000,3.655
min_duration = 5 s,385,5265,5265,0.001,3.576
min_duration = 20 s,385,5265,5265,0.001,1.852
min_duration = 60 s,385,5265,5265,0.000,1.377
min_duration = 300 s,385,5265,5265,0.001,1.243
min_duration = 1200 s,385,5262,5262,NaN,1.268


The windows are the same in both directions, and the receiver-transmitter windows are the same within the analysis of all 82 satellites. They are also the same (within 1 ms) for every `min_duration` from 5 to 300 s, since the shortest window within the day lasts nearly 8 minutes; at 1,200 s, 3 windows are missed, as documented. The analysis is fast: about 1.4 s for the 385 receiver-transmitter pairs over a day, and 3.9 s for all 6,642 pairs.

## Summary

| Test | Result |
| --- | --- |
| 1. Observed occultations | All 5,699 occultations reaching -80 km within the day at a window edge of the right kind (setting at a window's end, rising at its start) |
| 2. Timing | Edges within 0.25 to 0.30 s of the true times (95th percentile); a bias of 0.02 to 0.06 s early, specific to COSMIC-2 |
| 3. Consistency with `collect_ro_observations` | 10,019 occultations paired one to one within 1 ms; the remaining 54 edges and 39 occultations explained by grazing geometries and the ends of the day |
| 4. Symmetry, resolution, and cost | The same windows in both directions, within the analysis of all satellites, and for `min_duration` from 5 to 300 s; 1.4 s for 385 pairs over a day, 3.9 s for 6,642 |

**TAT-C's satellite-to-satellite occlusion is validated against COSMIC-2.** With published orbits, `collect_space_observations` places every occultation COSMIC-2 recorded on this day at the edge of a window of the right kind, within a few tenths of a second of the true time (limited by orbit accuracy), and agrees with `collect_ro_observations` to a millisecond, while analyzing all of a day's receiver-transmitter pairs in about a second.

**Findings for the RO analysis.** The comparison corrected the direction (`is_rising`) of `collect_ro_observations` for occultations nearly abeam of the receiver (Test 3), and documents two properties of its grazing occultations, outside COSMIC-2's yaw limit but within reach of receivers with wider antennas: it reports a single `time` for an observation whose straight line crosses -80 km twice, and it reports observations whose straight line never crosses -80 km with a `time` and `position` at the endpoint of the profile nearest -80 km.

**Next steps:**

1. *Slant range.* Validate `min_range` and `max_range` against precise orbits, for example the Level-1B orbits (GNV1B) of the two GRACE-FO satellites, about 200 km apart.
2. *Brief windows.* Test windows shorter than `min_duration`, such as close approaches between satellites (from CelesTrak's SOCRATES).
3. *Pointing limits.* `collect_space_observations` does not model the observing satellite's pointing limits, such as an antenna's field of view: its windows include the occultations outside COSMIC-2's yaw limit of 65 deg, which `collect_ro_observations` excludes with `max_yaw`.

**Limitations.** This study covers one day and one constellation of receivers in low-inclination orbits (24 deg). The occlusion constraint is tested at one altitude (-80 km) and without range limits, and the timing (Test 2) uses random samples of 200 occultations per constellation.